# Chapter 7: The six-sided die

In [1]:
import numpy as np
from expkit.sim.die import fair_die_rolls, loaded_die_rolls, face_counts, dirichlet_posterior_mean
from expkit.inference.chi2 import goodness_of_fit
from expkit.inference.binomial import binom_test_exact
from expkit.plot.style import apply_style
apply_style()

## Loop A: face counts wobble

In [2]:
for n in [60, 600, 6000]:
    counts = face_counts(fair_die_rolls(n, seed=70 + n))
    print(f'N={n:>5}  counts={counts.tolist()}  expected={n/6:.1f}')

N=   60  counts=[10, 9, 13, 9, 10, 9]  expected=10.0
N=  600  counts=[95, 98, 112, 106, 91, 98]  expected=100.0
N= 6000  counts=[1016, 971, 1044, 965, 1019, 985]  expected=1000.0


## Loop B: chi-square detects a loaded die

In [3]:
loaded_p = np.array([2/15]*5 + [1/3])
for n in [60, 200, 600, 2000]:
    fair_counts = face_counts(fair_die_rolls(n, seed=0))
    loaded_counts = face_counts(loaded_die_rolls(n, p=loaded_p, seed=0))
    print(f'N={n:>5}  fair p={goodness_of_fit(fair_counts).p_value:.4g}   loaded p={goodness_of_fit(loaded_counts).p_value:.4g}')

N=   60  fair p=0.5786   loaded p=0.0038
N=  200  fair p=0.3082   loaded p=5.921e-15
N=  600  fair p=0.3896   loaded p=7.838e-36
N= 2000  fair p=0.2548   loaded p=1.6e-88


## Loop C: Dirichlet posterior mean for a loaded-die observation

In [4]:
rolls = loaded_die_rolls(600, p=np.array([2/15]*5 + [1/3]), seed=71)
counts = face_counts(rolls)
post_mean = dirichlet_posterior_mean(counts)
print('observed counts:', counts.tolist())
print('posterior mean per face:', np.round(post_mean, 3).tolist())
print('uniform reference :', [round(1/6, 3)] * 6)

observed counts: [75, 89, 79, 79, 72, 206]
posterior mean per face: [0.125, 0.149, 0.132, 0.132, 0.12, 0.342]
uniform reference : [0.167, 0.167, 0.167, 0.167, 0.167, 0.167]


## Loop D: multiple comparisons trap

In [5]:
rng = np.random.default_rng(72)
naive_fp = 0; bonf_fp = 0; trials = 500
for _ in range(trials):
    rolls = fair_die_rolls(600, seed=int(rng.integers(0, 2**30)))
    counts = face_counts(rolls)
    pvals = [binom_test_exact(int(k), 600, p_null=1/6).p_value for k in counts]
    if any(p < 0.05 for p in pvals): naive_fp += 1
    if any(p < 0.05/6 for p in pvals): bonf_fp += 1
print(f'naive (alpha=0.05 per face): family-wise type-I rate = {naive_fp/trials:.3f}')
print(f'Bonferroni (alpha=0.0083 per face): family-wise type-I rate = {bonf_fp/trials:.3f}')

naive (alpha=0.05 per face): family-wise type-I rate = 0.214
Bonferroni (alpha=0.0083 per face): family-wise type-I rate = 0.046
